# MarkdownHeaderTextSplitter

`MarkdownHeaderTextSplitter`는 Markdown 문서를 **헤더 구조**에 따라 분할합니다.

## 주요 특징

- **구조 기반 분할**: `#`, `##`, `###` 등 헤더 레벨을 기준으로 분할
- **메타데이터 보존**: 각 청크에 헤더 정보를 메타데이터로 저장
- **계층적 정보**: 상위 헤더 정보를 하위 청크에도 유지

## 사용 시나리오

- ✅ **문서 구조 보존**: Markdown 문서의 계층 구조 유지
- ✅ **섹션별 검색**: 특정 섹션만 검색하거나 필터링
- ✅ **RAG 최적화**: 문서 구조 정보를 활용한 검색 품질 향상


# 06. MarkdownHeaderTextSplitter

## 학습 목표
- `MarkdownHeaderTextSplitter`로 `#`, `##`, `###` 헤더 기반으로 문서를 분할해요
- 헤더 정보가 메타데이터로 저장되는 방식을 이해해요
- `RecursiveCharacterTextSplitter`와 결합해서 2단계 분할을 구성해요

## 사전 지식
- 01-CharacterTextSplitter 완료
- Markdown 문법 기본 이해 (`#`, `##`, `###`)

---

> **왜 헤더 기반 분할이 유용한가?**
>
> 기술 문서를 예로 들면, "2.3 인증 방식" 섹션에 있는 내용임을 알면 검색 정확도가 높아져요. `MarkdownHeaderTextSplitter`는 이 헤더 정보를 **메타데이터로 자동 저장**해서 나중에 필터링이나 검색에 활용할 수 있게 해요.


> 🔑 **핵심 개념**: `MarkdownHeaderTextSplitter`의 핵심 가치는 헤더 정보를 **메타데이터로 저장**한다는 점입니다. 분할된 청크에 "이 내용은 '2.3 인증 방식' 섹션에 있다"는 정보가 자동으로 붙어서, 벡터 검색 후 출처 표시가 정확해집니다.

## 1. 기본 사용법


> 🎯 **강의 포인트**: 실무에서는 **2단계 분할 전략**을 많이 씁니다. ① `MarkdownHeaderTextSplitter`로 헤더 구조를 보존하며 섹션 분리 → ② 각 섹션이 너무 길면 `RecursiveCharacterTextSplitter`로 추가 분할. 헤더 메타데이터는 최종 청크에도 유지됩니다.

In [1]:
# ============================================================
# TODO: MarkdownHeaderTextSplitter로 Markdown 문서를 헤더 기준으로 분할해보세요
# 힌트: MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "Header 1"), ...]) → split_text(markdown_document)
# 예상 결과: 6개의 청크가 생성되고 각 청크의 메타데이터에 헤더 정보가 포함됩니다
# ============================================================

from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_document = """
# LangChain 가이드

## 소개

LangChain은 LLM 애플리케이션 개발 프레임워크입니다.

## 주요 기능

### 체인 (Chains)

여러 컴포넌트를 연결하여 복잡한 작업을 수행합니다.

### 에이전트 (Agents)

자율적으로 작업을 계획하고 실행합니다.

## 설치 방법

pip 명령어로 간단히 설치할 수 있습니다.

### 기본 설치

```bash
pip install langchain
```

### 추가 패키지

OpenAI 등 추가 패키지도 설치 가능합니다.
"""

# 1단계: 분할할 헤더 레벨 지정
# - 튜플 형식: (Markdown 헤더 기호, 메타데이터에 저장될 키 이름)
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

# 2단계: MarkdownHeaderTextSplitter 생성 및 분할
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_header_splits = markdown_splitter.split_text(markdown_document)

print(f"분할된 청크 개수: {len(md_header_splits)}")
for i, header in enumerate(md_header_splits):
    print(f"\n{'=' * 60}")
    print(f"[Chunk {i}]")
    print(f"메타데이터: {header.metadata}")
    print(f"내용: {header.page_content[:100]}...")
    print('=' * 60)

분할된 청크 개수: 6

[Chunk 0]
메타데이터: {'Header 1': 'LangChain 가이드', 'Header 2': '소개'}
내용: LangChain은 LLM 애플리케이션 개발 프레임워크입니다....

[Chunk 1]
메타데이터: {'Header 1': 'LangChain 가이드', 'Header 2': '주요 기능', 'Header 3': '체인 (Chains)'}
내용: 여러 컴포넌트를 연결하여 복잡한 작업을 수행합니다....

[Chunk 2]
메타데이터: {'Header 1': 'LangChain 가이드', 'Header 2': '주요 기능', 'Header 3': '에이전트 (Agents)'}
내용: 자율적으로 작업을 계획하고 실행합니다....

[Chunk 3]
메타데이터: {'Header 1': 'LangChain 가이드', 'Header 2': '설치 방법'}
내용: pip 명령어로 간단히 설치할 수 있습니다....

[Chunk 4]
메타데이터: {'Header 1': 'LangChain 가이드', 'Header 2': '설치 방법', 'Header 3': '기본 설치'}
내용: ```bash
pip install langchain
```...

[Chunk 5]
메타데이터: {'Header 1': 'LangChain 가이드', 'Header 2': '설치 방법', 'Header 3': '추가 패키지'}
내용: OpenAI 등 추가 패키지도 설치 가능합니다....


> ⚠️ **자주 하는 실수**: `strip_headers=True`(기본값)로 설정하면 헤더 텍스트가 `page_content`에서 제거됩니다. 헤더 내용이 임베딩 검색에 중요하다면 `strip_headers=False`로 설정하세요.

> 💡 **실무 팁**: API 문서, 기술 명세서, 위키 문서처럼 헤더 구조가 잘 잡힌 Markdown 파일에서 특히 효과적입니다. 헤더 없이 일반 산문체로 작성된 문서에는 큰 효과가 없습니다.

## 2. 헤더 유지 옵션

`strip_headers=False` 옵션으로 청크 내용에 헤더를 포함할 수 있습니다.


In [2]:
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False,  # 헤더를 내용에 포함
)

md_header_splits = markdown_splitter.split_text(markdown_document)

print(f"헤더 유지 모드 청크 개수: {len(md_header_splits)}")
print(f"\n첫 번째 청크:")
print(md_header_splits[0].page_content[:200])


헤더 유지 모드 청크 개수: 6

첫 번째 청크:
# LangChain 가이드  
## 소개  
LangChain은 LLM 애플리케이션 개발 프레임워크입니다.


## 핵심 정리 및 마무리

### 헤더 분할의 장점

| 기능 | 설명 |
|------|------|
| 구조 보존 | 문서의 계층 구조가 메타데이터에 저장됨 |
| 필터링 가능 | 특정 섹션만 검색하거나 필터링 가능 |
| 문맥 정보 | 상위 헤더 정보가 하위 청크에도 유지됨 |

### 2단계 분할 전략
> 실무에서는 **헤더 분할 → 크기 분할** 순서로 2단계를 적용해요. 먼저 문서 구조를 헤더로 나눈 다음, 각 섹션이 너무 길면 `RecursiveCharacterTextSplitter`로 추가 분할해요.

---

## 다음 예고

- **07-HTMLHeaderTextSplitter**: 웹 페이지의 `<h1>`, `<h2>` 태그 기반 분할
- **08-RecursiveJsonSplitter**: Chunking 섹션 마지막 — JSON 구조 보존 분할과 전략 선택 가이드
